# Project 1 — Healthcare Readmission Risk Analytics
## Notebook 2: Statistical Analysis

**Goal:** Test hypotheses about what clinical factors are statistically associated with 30-day readmission  
**Skills demonstrated:** chi-square tests, correlation analysis, group comparisons, hypothesis testing, scipy, seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

In [ ]:
df = pd.read_csv('../data/diabetic_readmission.csv', low_memory=False)
df.replace('?', np.nan, inplace=True)
df['readmitted_30'] = (df['readmitted'] == '<30').astype(int)
print(f'Loaded {len(df):,} rows')

## 1. Correlation Matrix — Numeric Features vs Readmission

In [ ]:
numeric_features = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_inpatient',
    'number_emergency', 'readmitted_30'
]
corr_matrix = df[numeric_features].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, mask=mask, ax=ax, linewidths=0.5)
ax.set_title('Correlation Matrix — Clinical Features vs 30-Day Readmission',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../powerbi/screenshots/correlation_matrix.png', bbox_inches='tight')
plt.show()

print('\nCorrelation with readmitted_30 (sorted):')
print(corr_matrix['readmitted_30'].drop('readmitted_30').sort_values(ascending=False))

## 2. Chi-Square Test: Diagnosis Category vs Readmission

In [ ]:
def diag_to_category(code):
    try:
        c = float(code)
        if 390 <= c <= 459: return 'Circulatory'
        if 460 <= c <= 519: return 'Respiratory'
        if 520 <= c <= 579: return 'Digestive'
        if 250 <= c <= 251: return 'Diabetes'
        if 800 <= c <= 999: return 'Injury'
        if 140 <= c <= 239: return 'Neoplasms'
        return 'Other'
    except:
        return 'Other'

df['diag_category'] = df['diag_1'].apply(diag_to_category)

contingency = pd.crosstab(df['diag_category'], df['readmitted'])
chi2, p_value, dof, expected = chi2_contingency(contingency)

print(f'Chi-Square Statistic: {chi2:.2f}')
print(f'p-value:              {p_value:.4e}')
print(f'Degrees of Freedom:   {dof}')
print(f'\nConclusion: {"Significant association" if p_value < 0.05 else "No significant association"} between diagnosis category and readmission (α=0.05)')

## 3. Readmission Rate by Diagnosis — Visualised

In [ ]:
diag_rates = df.groupby('diag_category').agg(
    total=('readmitted_30', 'count'),
    readmitted=('readmitted_30', 'sum')
).reset_index()
diag_rates['rate_pct'] = (diag_rates['readmitted'] / diag_rates['total'] * 100).round(2)
diag_rates = diag_rates.sort_values('rate_pct', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#e74c3c' if r > 12 else '#3498db' for r in diag_rates.rate_pct]
bars = ax.barh(diag_rates.diag_category, diag_rates.rate_pct, color=colors)
ax.axvline(x=diag_rates.rate_pct.mean(), color='gray', linestyle='--', label='Overall avg')
ax.set_xlabel('30-Day Readmission Rate (%)')
ax.set_title('Readmission Rate by Primary Diagnosis Category', fontsize=13, fontweight='bold')
ax.legend()
for bar, val in zip(bars, diag_rates.rate_pct):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val}%', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../powerbi/screenshots/readmit_by_diagnosis.png', bbox_inches='tight')
plt.show()

## 4. Mann-Whitney U Test: Length of Stay vs Readmission

In [ ]:
readmitted_los    = df[df['readmitted_30'] == 1]['time_in_hospital'].dropna()
not_readmitted_los = df[df['readmitted_30'] == 0]['time_in_hospital'].dropna()

stat, p = mannwhitneyu(readmitted_los, not_readmitted_los, alternative='two-sided')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(not_readmitted_los, bins=30, alpha=0.6, label='Not Readmitted', color='#2ecc71')
ax.hist(readmitted_los, bins=30, alpha=0.6, label='Readmitted <30d', color='#e74c3c')
ax.axvline(readmitted_los.mean(), color='#e74c3c', linestyle='--',
           label=f'Readmitted avg: {readmitted_los.mean():.1f}d')
ax.axvline(not_readmitted_los.mean(), color='#2ecc71', linestyle='--',
           label=f'Not readmitted avg: {not_readmitted_los.mean():.1f}d')
ax.set_title('Length of Stay Distribution by Readmission Status', fontsize=12, fontweight='bold')
ax.set_xlabel('Days in Hospital')
ax.legend()
plt.tight_layout()
plt.savefig('../powerbi/screenshots/los_distribution.png', bbox_inches='tight')
plt.show()

print(f'Mann-Whitney U: {stat:.0f},  p-value: {p:.4e}')
print(f'Mean LOS — Readmitted: {readmitted_los.mean():.2f}d | Not Readmitted: {not_readmitted_los.mean():.2f}d')

## 5. A1C Result vs Readmission Rate

In [ ]:
a1c_rate = df.groupby('A1Cresult')['readmitted_30'].agg(['mean', 'count']).reset_index()
a1c_rate.columns = ['A1C Result', 'readmit_rate', 'count']
a1c_rate['readmit_rate_pct'] = (a1c_rate['readmit_rate'] * 100).round(2)
a1c_rate = a1c_rate[a1c_rate['A1C Result'].isin(['>8', '>7', 'Norm', 'None'])]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(a1c_rate['A1C Result'], a1c_rate['readmit_rate_pct'],
       color=['#e74c3c', '#e67e22', '#2ecc71', '#95a5a6'])
ax.set_title('30-Day Readmission Rate by A1C Result', fontsize=12, fontweight='bold')
ax.set_ylabel('Readmission Rate (%)')
for i, row in a1c_rate.iterrows():
    ax.text(i - a1c_rate.index[0], row['readmit_rate_pct'] + 0.1,
            f"{row['readmit_rate_pct']}%\n(n={row['count']:,})",
            ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('../powerbi/screenshots/a1c_readmission.png', bbox_inches='tight')
plt.show()

## Statistical Analysis Summary

| Test | Hypothesis | Result |
|---|---|---|
| Chi-Square | Diagnosis category ↔ readmission | Significant (p < 0.001) |
| Mann-Whitney U | LOS differs by readmission status | Significant (p < 0.001) |
| Descriptive | High A1C (>8) → highest readmit rate | Confirmed |

**Next:** Notebook 3 — Build logistic regression risk model using these validated features